# 04 — Retrospective evaluation and comparison

The four existing filenames are retained. This revision replaces the previous feature-ranking experiment with the fixed V63 business feature list. The folder name is retained for compatibility; no feature selection runs here. Do not regenerate these notebooks with the older experiment build script or use the old folder README as this revision's run guide.

**Source blocker:** the repository contains snapshot values/readers, but no verified V63 historical feature-generation SQL or patient observation-coverage rule. The dictionary explains meaning, not every calculation. Notebook 01 displays the audit and stops until the missing calculations and coverage are supplied in its reconstruction cell. No historical values, coverage percentages or performance are claimed in this delivery.

Existing population: 23,151 patient snapshots, 12,447 patients, 1,345 positives; PATIENT_ID + END_DT; labels copied unchanged from the frozen source. The model-type FEATURES array supplies all 49 predictors in stored order. Calendar position 0 is newest; monthly positions 0–11 cover the original calendar buckets, with the current month truncated at END_DT. Quarterly positions 0–3 group exactly those buckets into consecutive three-month periods, not calendar-year quarters. Historical rolling windows can require source records earlier than this displayed 12-bucket sequence.

Architecture and optimizer remain the original temporal Transformer (128 width, 4 heads, 2 layers, FF256, dropout .2; weighted BCE, AdamW, validation-AP checkpointing). Mixed business values use the existing 49-feature model's TRAIN-only median/mean/std preprocessing principle rather than applying the old raw-count log1p. Padded rows are excluded from fitted statistics and re-zeroed afterwards. No feature is silently imputed to resolve missing reconstruction logic.

Only aggregate reports, tensors and model checkpoints are saved in the existing private warehouse pattern. No custom CSV/download export is provided. Clear all outputs before committing executed notebooks. TEST has previously been inspected and remains a retrospective check. Monthly/quarterly and masking are hypotheses; this two-model comparison alone does not isolate a causal masking effect from representation/preprocessing changes.


In [ ]:
# Connection and matching run identifiers
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
if 'sf_options' not in globals() or not isinstance(sf_options, dict) or 'spark' not in globals():
    raise RuntimeError('Supply the existing private sf_options connection on the approved Spark runtime.')
sf_options_dl_poc = dict(sf_options)
DATABASE = 'DSVC_TAKEDA_TA_PRIVATE'
sf_options_dl_poc.update(sfDatabase=DATABASE, sfSchema='DS_ML')
SOURCE_PREFIX = 'TAK861_TX_READY_V63'
PREFIX = SOURCE_PREFIX + '_DL_POC'
DATASET_ID = 'H001'
RUN_ID = 'M001'
import re
if any(not re.fullmatch(r'[A-Z][A-Z0-9_]{0,15}', v) for v in (DATASET_ID, RUN_ID)):
    raise ValueError('Use short uppercase identifiers; use matching IDs in all four notebooks.')
# New output names prevent collision with prior saved models; original inputs remain unchanged.
EXPERIMENT_PREFIX = PREFIX + '_BUSINESS49_TEMPORAL_V1'
PREPARED_TABLE = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_INPUTS'
SPLIT_TABLE = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_SPLIT'
RUN_PREFIX = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_' + RUN_ID
REFERENCE_MODEL_TABLE = PREFIX + '_MODEL_RUN_001'
REFERENCE_NAMES = {'checkpoint.pt', 'training_summary.json', 'training_history.csv', 'training_history.png'}
MODEL_NAMES = {'checkpoint.pt', 'summary.json', 'history.json'}
MODEL_SETTINGS = dict(d_model=128, n_heads=4, encoder_layers=2, feedforward_dim=256, dropout=.2)
TRAINING_SETTINGS = dict(seed=42, epochs=20, patience=5, min_delta=1e-4, batch_size=64,
                        learning_rate=.001, weight_decay=.0001, grad_clip=1., device='auto')
print('Fixed V63 business features; source cohort and RESP retained. Run MONTHLY then QUARTERLY.')

IMPLEMENTATION_SHA256 = 'eba7968a1cfa16f62857d5a220a5fda1a75fcbb3bb8faebc01987abcfb7978f1'


In [ ]:
# Embedded verified checkpoint and evaluation helpers
"""Embedded notebook helpers: fixed business features, calendar grids and padding."""
import io
import json
import hashlib
import re
import marshal
import numpy as np
import pandas as pd


def require(condition, message):
    if not condition:
        raise ValueError(message)


def ordered_features():
    features, comparison = configured_features()
    require(len(features) == 49, 'V63 MODEL_TYPE.FEATURES must contain exactly 49 predictors.')
    return features, comparison


def snapshot_records(metadata):
    return [[r.PATIENT_ID, r.END_DT, int(r.RESP)] for r in metadata.itertuples()]


def array_hash(values):
    values = np.asarray(values)
    h = hashlib.sha256(canonical_json(list(values.shape)).encode())
    h.update(np.isnan(values).astype('u1').tobytes())
    h.update(np.nan_to_num(values, nan=0).astype('<f8').tobytes())
    return h.hexdigest()


def sequence_grid(metadata, representation):
    require(representation in ('MONTHLY', 'QUARTERLY'), 'Unknown representation.')
    width = 1 if representation == 'MONTHLY' else 3
    rows = []
    for r in metadata.itertuples():
        cutoff = pd.Timestamp(r.END_DT)
        month = cutoff.to_period('M')
        for step in range(12 // width):
            start = (month - width * step - (width - 1)).start_time.normalize()
            end = min(cutoff, (month - width * step).end_time.normalize())
            rows.append((r.PATIENT_ID, r.END_DT, step, start.strftime('%Y-%m-%d'),
                         end.strftime('%Y-%m-%d'), int(r.RESP)))
    grid = pd.DataFrame(rows, columns=['PATIENT_ID', 'END_DT', 'TIME_STEP', 'PERIOD_START', 'PERIOD_END', 'RESP'])
    require(not grid.duplicated(['PATIENT_ID', 'END_DT', 'TIME_STEP']).any(), 'Duplicate sequence keys.')
    require(grid.PERIOD_END.le(grid.END_DT).all(), 'Future period boundary.')
    return grid


def verify_periods(monthly, quarterly):
    keys = ['PATIENT_ID', 'END_DT']
    for frame, count in ((monthly, 12), (quarterly, 4)):
        require(not frame.duplicated(keys + ['TIME_STEP']).any(), 'Duplicate sequence row.')
        require(frame.groupby(keys).TIME_STEP.apply(lambda x: sorted(x) == list(range(count))).all(),
                'Missing or invalid sequence positions.')
    for q in range(4):
        m = monthly.loc[monthly.TIME_STEP.between(q * 3, q * 3 + 2)]
        bounds = m.groupby(keys).agg(PERIOD_START=('PERIOD_START', 'min'), PERIOD_END=('PERIOD_END', 'max'))
        actual = quarterly.loc[quarterly.TIME_STEP.eq(q)].set_index(keys)[['PERIOD_START', 'PERIOD_END']].sort_index()
        require(bounds.sort_index().equals(actual), 'Monthly and quarterly calendar periods differ.')


def reconcile_dictionary(features, dictionary):
    rows, assigned = [], set()
    for order, name in enumerate(features):
        exact = [r for r in dictionary if r['name_complete'] and r['name'] == name]
        candidates = exact or [r for r in dictionary if not r['name_complete'] and name.startswith(r['name'])]
        match = candidates[0] if len(candidates) == 1 else None
        if match:
            require(match['seq'] not in assigned, 'Dictionary entry matched multiple model features; confirm full names.')
            assigned.add(match['seq'])
        rows.append({'FEATURE_ORDER': order, 'FEATURE_NAME': name,
                     'DICTIONARY_SEQ': match['seq'] if match else None,
                     'MATCH': ('EXACT_NAME' if exact else 'UNIQUE_VISIBLE_PREFIX') if match else 'UNRESOLVED',
                     'BUSINESS_DEFINITION': match['definition'] if match else 'No unambiguous dictionary match.',
                     'FEATURE_TYPE': (match['type_label'] + ' (dictionary label; not a casting rule)') if match else 'UNRESOLVED',
                     'DEFAULT_STATUS': match['default_status'] if match else 'UNRESOLVED',
                     'NOTES': ((match.get('notes', '') + ('; clipped name matched by unique visible prefix' if not exact else ''))
                               if match else 'Confirm full feature name/definition against V63.')})
    unmatched = pd.DataFrame([r for r in dictionary if r['seq'] not in assigned])
    return pd.DataFrame(rows), unmatched


def reconstruction_audit(features, dictionary, rules):
    matched, extra = reconcile_dictionary(features, dictionary)
    require(not set(rules).difference(features), 'Historical rules include non-V63 features.')
    rows = []
    for row in matched.to_dict('records'):
        name = row['FEATURE_NAME']
        rule = rules.get(name)
        row['SOURCE/CALCULATION'] = SOURCE_PREFIX + '_MODEL_DATA; snapshot values only verified locally'
        row['HISTORICAL_RECONSTRUCTION_STATUS'] = row.pop('DEFAULT_STATUS')
        row['HISTORICAL_LOGIC'] = 'Historical V63 SQL and observation coverage not supplied; no replication or zero substitution.'
        if rule:
            status = rule.get('status')
            require(status in ('EXACT', 'APPROXIMATED', 'SNAPSHOT_ONLY', 'UNRESOLVED'), 'Invalid reconstruction status.')
            row['HISTORICAL_RECONSTRUCTION_STATUS'] = status
            row['SOURCE/CALCULATION'] = rule.get('source', '')
            row['HISTORICAL_LOGIC'] = rule.get('logic', '')
            row['NOTES'] += '; ' + rule.get('notes', '')
            if status in ('EXACT', 'APPROXIMATED'):
                require(all(rule.get(k) for k in ('source', 'logic', 'evidence', 'observation_logic')),
                        name + ': source, calculation evidence and observation logic are required.')
                require(callable(rule.get('builder')), name + ': executable historical calculation is missing.')
                if status == 'APPROXIMATED':
                    require(bool(rule.get('approximation')), name + ': describe the approximation explicitly.')
        rows.append(row)
    audit = pd.DataFrame(rows)
    counts = audit.HISTORICAL_RECONSTRUCTION_STATUS.value_counts().reindex(
        ['EXACT', 'APPROXIMATED', 'SNAPSHOT_ONLY', 'UNRESOLVED'], fill_value=0)
    require(len(audit) == int(counts.sum()) == 49, 'Reconstruction counts must sum to 49.')
    return audit, counts, extra


def require_reconstruction(audit, rules):
    blocked = audit.loc[~audit.HISTORICAL_RECONSTRUCTION_STATUS.isin(['EXACT', 'APPROXIMATED']), 'FEATURE_NAME'].tolist()
    require(not blocked, 'Historical reconstruction blocked. Supply verified V63 calculations and coverage for: ' + ', '.join(blocked))
    require(set(rules) == set(audit.FEATURE_NAME), 'Exactly 49 historical rules are required.')


def construct_sequence(grid, features, audit, rules, representation):
    require_reconstruction(audit, rules)
    keys = ['PATIENT_ID', 'END_DT', 'TIME_STEP']
    # Builders receive dates and identifiers only, never RESP or split membership.
    request = grid.drop(columns='RESP').copy()
    values, observed = [], []
    provenance = []
    for feature in features:
        rule = rules[feature]
        result = rule['builder'](request.copy(), representation)
        needed = keys + ['VALUE', 'IS_OBSERVED', 'MAX_EVENT_DATE', 'MAX_AVAILABLE_DATE',
                         'OBSERVATION_EVIDENCE', 'PROVENANCE_KIND', 'PROVENANCE_NOTE']
        require(isinstance(result, pd.DataFrame) and set(needed).issubset(result.columns), feature + ': incomplete historical output.')
        result = result[needed].copy()
        require(not result[keys].isna().any().any() and not result.duplicated(keys).any(), feature + ': invalid historical keys.')
        require(len(result) == len(grid), feature + ': historical output must cover every requested position explicitly.')
        aligned = request.merge(result, on=keys, how='left', validate='one_to_one', indicator=True)
        require(aligned._merge.eq('both').all(), feature + ': missing historical keys.')
        require(aligned.IS_OBSERVED.isin([0, 1, False, True]).all(), feature + ': observation status is required for every position.')
        known = aligned.IS_OBSERVED.astype(bool).to_numpy()
        require(aligned.OBSERVATION_EVIDENCE.map(lambda v: isinstance(v, str) and bool(v.strip())).all(),
                feature + ': availability must be evidenced, including unavailable periods.')
        for field in ('MAX_EVENT_DATE', 'MAX_AVAILABLE_DATE'):
            dates = pd.to_datetime(aligned[field], errors='raise')
            require(dates.dt.tz is None and dates.dropna().eq(dates.dropna().dt.normalize()).all(),
                    feature + ': provenance dates must be exact dates; timestamp rules require explicit review.')
            require((dates.isna() | dates.le(pd.to_datetime(aligned.PERIOD_END))).all(), feature + ': future information detected in ' + field)
        numbers = pd.to_numeric(aligned.VALUE, errors='raise').to_numpy(dtype=np.float64)
        require(np.isfinite(numbers[known]).all(), feature + ': observed values must be finite; explicitly calculate observed zeros.')
        require(np.isnan(numbers[~known]).all(), feature + ': unavailable feature history must remain null before padding.')
        kinds = aligned.PROVENANCE_KIND
        require(kinds.isin(['EVENT_DERIVED', 'OBSERVED_EMPTY', 'STATIC', 'UNAVAILABLE']).all(), feature + ': invalid provenance kind.')
        require(aligned.PROVENANCE_NOTE.map(lambda v: isinstance(v, str) and bool(v.strip())).all(), feature + ': missing provenance explanation.')
        require(np.array_equal(kinds.ne('UNAVAILABLE').to_numpy(), known), feature + ': provenance and availability disagree.')
        event_rows = kinds.eq('EVENT_DERIVED')
        require(aligned.loc[event_rows, ['MAX_EVENT_DATE', 'MAX_AVAILABLE_DATE']].notna().all().all(),
                feature + ': event-derived values need both event and availability date maxima.')
        require(np.all(numbers[kinds.eq('OBSERVED_EMPTY')] == 0), feature + ': observed-empty provenance requires a defined zero value.')
        if kinds.eq('STATIC').any():
            require(rule.get('static_feature') is True and bool(rule.get('static_rationale')),
                    feature + ': static provenance requires a verified static-feature rule and rationale.')
        values.append(numbers)
        observed.append(known)
        provenance.append({'feature': feature, 'rule': {k: v for k, v in rule.items() if k != 'builder'},
                           'builder_sha256': hashlib.sha256(marshal.dumps(rule['builder'].__code__)).hexdigest(),
                           'observed_positions': int(known.sum())})
    raw = np.column_stack(values)
    known = np.column_stack(observed)
    # A token is fully observed only when all fixed 49 inputs are supported at this cutoff.
    # Partial feature availability is reported, not disguised as no activity.
    valid = known.all(axis=1)
    long = grid.copy()
    long[features] = raw
    long['AVAILABLE_FEATURE_COUNT'] = known.sum(axis=1)
    long['IS_VALID_TIMESTEP'] = valid.astype('int64')
    long['IS_PADDED'] = (~valid).astype('int64')
    long['PADDING_REASON'] = np.where(valid, '', 'Insufficient evidenced history for one or more fixed features')
    # Preserve raw missingness for review; only the model tensor receives padding zeros.
    n_steps = 12 if representation == 'MONTHLY' else 4
    X = np.where(valid[:, None], raw, 0).astype(np.float32).reshape(-1, n_steps, 49)
    mask = valid.reshape(-1, n_steps)
    require(np.isfinite(X).all(), 'NaN/Inf after padding.')
    require(np.array_equal(mask, long.IS_VALID_TIMESTEP.to_numpy().reshape(mask.shape)), 'Mask alignment failure.')
    observed_zero = known.all(axis=1) & np.all(raw == 0, axis=1)
    require(valid[observed_zero].all(), 'Observed zero activity was incorrectly masked.')
    return {'X': X, 'valid': mask, 'long': long, 'known': known, 'provenance': provenance,
            'representation': representation}


def sparsity_report(bundle, features):
    raw = bundle['long'][features].to_numpy(dtype=float)
    known = bundle['known']
    valid = bundle['valid']
    counts = known.sum(axis=0)
    zeros = ((raw == 0) & known).sum(axis=0)
    feature = pd.DataFrame({'FEATURE_NAME': features, 'OBSERVED_VALUES': counts,
                            'UNAVAILABLE_VALUES': (~known).sum(axis=0), 'OBSERVED_ZERO_VALUES': zeros,
                            'OBSERVED_ZERO_PERCENT': np.divide(100. * zeros, counts, out=np.full(49, np.nan), where=counts > 0)})
    eligible = int(known.sum())
    report = {'MODEL': bundle['representation'], 'TOTAL_TIMESTEPS': int(valid.size),
              'VALID_TIMESTEPS': int(valid.sum()), 'PADDED_TIMESTEPS': int((~valid).sum()),
              'VALID_PERCENT': float(valid.mean() * 100), 'PADDED_PERCENT': float((~valid).mean() * 100),
              'ALL_PADDED_SNAPSHOTS': int((~valid.any(axis=1)).sum()),
              'OBSERVED_FEATURE_ZERO_PERCENT': float(((raw == 0) & known).sum() * 100 / eligible) if eligible else None,
              'TENSOR_ZERO_PERCENT_INCLUDING_PADDING': float((bundle['X'] == 0).mean() * 100),
              'OBSERVED_ZERO_TIMESTEPS': int((valid.reshape(-1) & np.all(raw == 0, axis=1)).sum())}
    distribution = pd.Series(valid.sum(axis=1)).value_counts().sort_index().rename_axis('VALID_TIMESTEPS').reset_index(name='SNAPSHOTS')
    return report, feature, distribution


def display_sequence(bundle, features):
    frame = bundle['long'].copy()
    if bundle['representation'] == 'MONTHLY':
        frame = frame.rename(columns={'PERIOD_START': 'MONTH_START', 'PERIOD_END': 'MONTH_END'})
    else:
        frame = frame.rename(columns={'TIME_STEP': 'QUARTER_TIME_STEP', 'PERIOD_START': 'QUARTER_START', 'PERIOD_END': 'QUARTER_END'})
    print(bundle['representation'], 'raw historical values; unavailable values are null here and zero-padded only in the model tensor')
    for label in (0, 1):
        sample = frame.loc[frame.RESP.eq(label)].head(24)
        if not sample.empty:
            print('RESP =', label)
            display(sample)
    for state in (1, 0):
        sample = frame.loc[frame.IS_VALID_TIMESTEP.eq(state)].head(12)
        print('Valid' if state else 'Padded', 'positions:', 'available' if not sample.empty else 'none in this dataset')
        if not sample.empty:
            display(sample)
    raw = bundle['long']
    summary = raw.groupby(['PATIENT_ID', 'END_DT'], sort=True).agg(
        VALID=('IS_VALID_TIMESTEP', 'sum'), PADDED=('IS_PADDED', 'sum')).reset_index()
    activity = raw[features].fillna(0).ne(0).sum(axis=1)
    summary['NONZERO_VALUES'] = activity.groupby([raw.PATIENT_ID, raw.END_DT]).sum().to_numpy()
    choices = [('relatively dense', summary.sort_values(['VALID', 'NONZERO_VALUES'], ascending=False).head(1)),
               ('partially sparse', summary.loc[summary.VALID.gt(0)].sort_values('NONZERO_VALUES').head(1)),
               ('substantial padding', summary.loc[summary.PADDED.gt(0)].sort_values('PADDED', ascending=False).head(1))]
    for title, example in choices:
        print('Structural example:', title, '(selected without model scores)')
        if example.empty:
            print('No matching example in this dataset.')
        else:
            r = example.iloc[0]
            display(frame.loc[frame.PATIENT_ID.eq(r.PATIENT_ID) & frame.END_DT.eq(r.END_DT)])


def prepared_blobs(metadata, features, bundles, audit, comparison, snapshot_X):
    report = {'schema': 2, 'dataset_id': DATASET_ID, 'features': features,
              'population_sha256': digest_json(snapshot_records(metadata)),
              'snapshot_feature_sha256': array_hash(snapshot_X),
              'configuration_audit': comparison, 'audit': audit.astype(object).where(pd.notna(audit), None).to_dict('records'),
              'orientation': '0=newest; calendar buckets; most recent bucket truncated at END_DT',
              'implementation_sha256': IMPLEMENTATION_SHA256,
              'representations': {}}
    artifacts = {'population.json': canonical_json(snapshot_records(metadata)).encode()}
    for name, b in bundles.items():
        buffer = io.BytesIO()
        np.savez_compressed(buffer, X=b['X'], valid=b['valid'])
        artifacts[name + '.npz'] = buffer.getvalue()
        sparsity, _, distribution = sparsity_report(b, features)
        report['representations'][name] = {'shape': list(b['X'].shape), 'X_sha256': array_hash(b['X']),
              'valid_sha256': array_hash(b['valid']), 'sparsity': sparsity,
              'valid_distribution': distribution.to_dict('records'), 'provenance': b['provenance']}
    artifacts['manifest.json'] = canonical_json(report).encode()
    return artifacts


PREPARED_NAMES = {'population.json', 'manifest.json', 'MONTHLY.npz', 'QUARTERLY.npz'}
SPLIT_NAMES = {'split.json', 'preprocessing.json', 'audit.json'}


def load_prepared():
    blobs = read_artifacts(PREPARED_TABLE, PREPARED_NAMES)
    manifest = json.loads(blobs['manifest.json'])
    require(manifest['schema'] == 2 and manifest['dataset_id'] == DATASET_ID, 'Prepared dataset ID/version changed.')
    require(manifest['implementation_sha256'] == IMPLEMENTATION_SHA256, 'Prepared dataset was produced by different code.')
    features, _ = ordered_features()
    require(features == manifest['features'], 'Authoritative feature order changed after preparation.')
    metadata = pd.DataFrame(json.loads(blobs['population.json']), columns=['PATIENT_ID', 'END_DT', 'RESP'])
    metadata = normalize_metadata(metadata).sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
    population_check(metadata)
    source = normalize_metadata(read_table(PREFIX + '_SNAPSHOTS').select('PATIENT_ID', 'END_DT', 'RESP').toPandas()).sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
    require(metadata.equals(source), 'Frozen population/RESP changed after preparation.')
    require(digest_json(snapshot_records(metadata)) == manifest['population_sha256'], 'Population hash mismatch.')
    audit = pd.DataFrame(manifest['audit'])
    require(audit.FEATURE_NAME.tolist() == features and audit.HISTORICAL_RECONSTRUCTION_STATUS.isin(['EXACT', 'APPROXIMATED']).all(), 'Unsupported or misordered reconstruction audit.')
    bundles = {}
    for name, count in (('MONTHLY', 12), ('QUARTERLY', 4)):
        with np.load(io.BytesIO(blobs[name + '.npz']), allow_pickle=False) as arrays:
            X, valid = arrays['X'].copy(), arrays['valid'].copy()
        require(X.shape == (len(metadata), count, 49) and valid.shape == X.shape[:2] and valid.dtype == bool, 'Invalid tensor or mask dimensions.')
        require(np.isfinite(X).all() and np.all(X[~valid] == 0), 'Invalid padding or nonfinite input.')
        require(array_hash(X) == manifest['representations'][name]['X_sha256'] and array_hash(valid) == manifest['representations'][name]['valid_sha256'], 'Tensor/mask fingerprint changed.')
        bundles[name] = {'raw_X': X, 'valid': valid, 'representation': name}
    return metadata, features, bundles, manifest


def fit_temporal_preprocessor(X, valid, rows):
    observed = X[rows][valid[rows]]
    require(len(observed) > 0, 'No observed TRAIN timesteps; cannot fit preprocessing.')
    require(np.isfinite(observed).all(), 'Unresolved missingness cannot be imputed as reconstructed history.')
    state = fit_preprocessor(observed)
    return state


def transform_temporal(X, valid, state):
    values, _ = transform_features(X.reshape(-1, X.shape[-1]), state)
    values = values.reshape(X.shape)
    values[~valid] = 0
    require(np.isfinite(values).all(), 'Nonfinite Transformer input.')
    return values


def split_statistics(metadata):
    out = []
    sets = {name: set(metadata.loc[metadata.SPLIT.eq(name), 'PATIENT_ID']) for name in ('train', 'validation', 'test')}
    expected = {'train': (16256, 8712, 941), 'validation': (3481, 1867, 202), 'test': (3414, 1868, 202)}
    for name, ids in sets.items():
        part = metadata.loc[metadata.SPLIT.eq(name)]
        require((len(part), len(ids), int(part.RESP.sum())) == expected[name], 'Original split counts changed: ' + name)
        out.append({'SPLIT': name, 'PATIENTS': len(ids), 'SNAPSHOTS': len(part), 'RESP_0': int(part.RESP.eq(0).sum()),
                    'RESP_1': int(part.RESP.sum()), 'POSITIVE_RATE': float(part.RESP.mean())})
    for a, b in (('train', 'validation'), ('train', 'test'), ('validation', 'test')):
        require(not sets[a].intersection(sets[b]), 'Patient overlap: ' + a + '/' + b)
    return pd.DataFrame(out)


def load_experiment():
    metadata, features, bundles, manifest = load_prepared()
    blobs = read_artifacts(SPLIT_TABLE, SPLIT_NAMES)
    audit = json.loads(blobs['audit.json'])
    states = json.loads(blobs['preprocessing.json'])
    frozen = read_table(PREFIX + '_PATIENT_SPLIT').select('PATIENT_ID', 'END_DT', 'RESP', 'SPLIT', 'SPLIT_CONFIG').toPandas()
    reference = json.loads(read_artifacts(REFERENCE_MODEL_TABLE, REFERENCE_NAMES)['training_summary.json'])
    metadata, hashes = bind_split(metadata, frozen, reference)
    split_statistics(metadata)
    require(hashes == audit['reference_hashes'] and digest_json(manifest) == audit['manifest_sha256'], 'Saved split audit changed.')
    records = [[r.PATIENT_ID, r.END_DT, int(r.RESP), r.SPLIT] for r in metadata.itertuples()]
    require(records == json.loads(blobs['split.json']), 'Saved assignments changed.')
    require(digest_json(states) == audit['preprocessing_sha256'], 'Saved preprocessing changed.')
    indices = {s: np.flatnonzero(metadata.SPLIT.eq(s).to_numpy()) for s in ('train', 'validation', 'test')}
    experiments = {}
    for name, b in bundles.items():
        state = states[name]
        require(state == fit_temporal_preprocessor(b['raw_X'], b['valid'], indices['train']), 'Preprocessing does not reproduce TRAIN-only fit.')
        X = transform_temporal(b['raw_X'], b['valid'], state)
        experiments[name] = dict(b, X=X, y=metadata.RESP.to_numpy(dtype=np.float32), metadata=metadata,
            indices=indices, features=features, preprocessor=state,
            hashes={'manifest_sha256': digest_json(manifest), 'snapshot_manifest_sha256': hashes['snapshot_manifest_sha256'],
                    'preprocessing_sha256': digest_json(state), 'model_input_sha256': array_hash(X), 'mask_sha256': array_hash(b['valid'])})
    require(np.array_equal(experiments['MONTHLY']['y'], experiments['QUARTERLY']['y']), 'Representation labels differ.')
    return experiments, manifest, audit

def canonical_json(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)

def digest_json(value):
    return hashlib.sha256(canonical_json(value).encode()).hexdigest()

def parse_features(value):
    if isinstance(value, str):
        value = json.loads(value)
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if not isinstance(value, list) or not value or any(not isinstance(x, str) or not x.strip() for x in value):
        raise ValueError("FEATURES must be a nonempty array of exact column names.")
    if len(set(x.upper() for x in value)) != len(value):
        raise ValueError("Duplicate feature names in MODEL_TYPE.FEATURES.")
    prohibited = {"PATIENT_ID", "START_DT", "END_DT", "RESP", "SPLIT", "RND", "SCORE", "DECILE", "CENTILE", "MILLILE"}
    if prohibited.intersection(x.upper() for x in value):
        raise ValueError("The selected list contains an identifier, target, split, random helper or prediction output.")
    return value

def quote_identifier(name):
    return '"' + name.replace('"', '""') + '"'

def normalize_metadata(frame):
    out = frame[["PATIENT_ID", "END_DT", "RESP"]].copy()
    if out.empty or out.isna().any().any():
        raise ValueError("Missing snapshot keys or labels.")
    if not out.PATIENT_ID.map(lambda x: isinstance(x, str) and bool(x.strip())).all():
        raise ValueError("Patient IDs must remain nonempty strings.")
    dates = pd.to_datetime(out.END_DT, errors="raise")
    if dates.dt.tz is not None or not dates.eq(dates.dt.normalize()).all():
        raise ValueError("Snapshot cutoffs must be exact dates.")
    out["END_DT"] = dates.dt.strftime("%Y-%m-%d")
    if not out.RESP.isin([0, 1]).all():
        raise ValueError("Nonbinary labels.")
    out["RESP"] = out.RESP.astype("int64")
    if out.duplicated(["PATIENT_ID", "END_DT"]).any():
        raise ValueError("Duplicate patient/date keys; no automatic deduplication is permitted.")
    return out

def align_features(snapshots, model_data, features):
    features = parse_features(features)
    expected = normalize_metadata(snapshots).sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)
    actual = normalize_metadata(model_data)
    missing = set(features).difference(model_data.columns)
    if missing:
        raise ValueError("Selected columns absent from MODEL_DATA: " + repr(sorted(missing)))
    source = actual.copy()
    for name in features:
        # Decimal fractions are converted to float, never through an integer cast.
        source[name] = pd.to_numeric(model_data[name], errors="raise").to_numpy(dtype=np.float64)
    aligned = expected.merge(source, on=["PATIENT_ID", "END_DT"], how="left",
                             validate="one_to_one", suffixes=("", "_SOURCE"), indicator=True)
    if not aligned._merge.eq("both").all() or not aligned.RESP.eq(aligned.RESP_SOURCE).all():
        raise ValueError("Missing source keys or conflicting labels in MODEL_DATA.")
    X = aligned[features].to_numpy(dtype=np.float64)
    if np.isinf(X).any():
        raise ValueError("Infinite source feature values.")
    return expected, X

def bind_split(metadata, frozen, reference):
    original = normalize_metadata(metadata).sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)
    normalized = normalize_metadata(frozen)
    normalized["SPLIT"] = frozen.SPLIT.to_numpy()
    normalized["SPLIT_CONFIG"] = frozen.SPLIT_CONFIG.to_numpy()
    normalized = normalized.sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)
    if not original.equals(normalized[["PATIENT_ID", "END_DT", "RESP"]]):
        raise ValueError("Saved split differs from the prepared snapshots/labels.")
    if normalized[["SPLIT", "SPLIT_CONFIG"]].isna().any().any():
        raise ValueError("Incomplete frozen split.")
    if set(normalized.SPLIT) != {"train", "validation", "test"}:
        raise ValueError("Unexpected split names.")
    if normalized.groupby("PATIENT_ID").SPLIT.nunique().gt(1).any():
        raise ValueError("Patient leakage between splits.")
    if normalized.SPLIT_CONFIG.nunique() != 1:
        raise ValueError("Inconsistent split configuration.")
    records = [[r.PATIENT_ID, r.END_DT, int(r.RESP), r.SPLIT] for r in normalized.itertuples()]
    hashes = {"snapshot_manifest_sha256": digest_json(records),
              "split_config_sha256": digest_json(json.loads(normalized.SPLIT_CONFIG.iloc[0]))}
    if reference.get("run_id") != "RUN_001" or reference.get("training_complete") is not True:
        raise ValueError("Expected completed original RUN_001 reference.")
    if any(reference.get("input_hashes", {}).get(k) != v for k, v in hashes.items()):
        raise ValueError("Patient assignments differ from original RUN_001 fingerprints.")
    for _, part in normalized.groupby("SPLIT"):
        if set(part.RESP) != {0, 1}:
            raise ValueError("Each split needs both outcome classes.")
    return normalized, hashes

def fit_preprocessor(X_train):
    if X_train.ndim != 2 or not len(X_train) or np.isinf(X_train).any():
        raise ValueError("Invalid training feature matrix.")
    all_missing = np.isnan(X_train).all(axis=0)
    median = np.array([0.0 if missing else np.nanmedian(X_train[:, i])
                       for i, missing in enumerate(all_missing)])
    filled = np.where(np.isnan(X_train), median, X_train)
    mean = filled.mean(axis=0)
    scale = filled.std(axis=0)
    scale[scale == 0] = 1.0
    if not np.isfinite(np.r_[median, mean, scale]).all():
        raise ValueError("Nonfinite preprocessing statistics.")
    return {"median": median.tolist(), "mean": mean.tolist(), "scale": scale.tolist(),
            "all_missing_train": all_missing.tolist()}

def transform_features(X, state):
    if X.ndim != 2 or X.shape[1] != len(state["median"]) or np.isinf(X).any():
        raise ValueError("Feature shape or values changed.")
    mask = np.isnan(X)
    values = ((np.where(mask, state["median"], X) - state["mean"]) / state["scale"]).astype(np.float32)
    if not np.isfinite(values).all():
        raise ValueError("Nonfinite standardized values.")
    return values, mask.astype(np.float32)
def read_table(table):
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load())

def read_query(query):
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("query", query).load())

def configured_features():
    rows = read_table(SOURCE_PREFIX + "_MODEL_TYPE").select("MODEL_TYPE", "FEATURES").collect()
    if len(rows) != 1:
        raise ValueError("Expected exactly one MODEL_TYPE configuration row.")
    features = parse_features(rows[0]["FEATURES"])
    summary = read_table(SOURCE_PREFIX + "_FINAL_MODEL").toPandas()
    if "FEATURES" not in summary.columns:
        raise ValueError("FINAL_MODEL lacks FEATURES.")
    summarized = summary.FEATURES.tolist()
    if any(not isinstance(f, str) or not f for f in summarized):
        raise ValueError("Invalid FINAL_MODEL feature name.")
    comparison = {"configured_model_type": str(rows[0]["MODEL_TYPE"]),
                  "configured_feature_count": len(features), "final_summary_rows": len(summary),
                  "configured_not_in_summary": sorted(set(features) - set(summarized)),
                  "summary_not_in_configuration": sorted(set(summarized) - set(features)),
                  "summary_duplicate_names": sorted(summary.loc[summary.FEATURES.duplicated(), "FEATURES"].unique().tolist()),
                  "rule": "MODEL_TYPE.FEATURES is authoritative, in its stored order; FINAL_MODEL is an audit."}
    columns = set(read_table(SOURCE_PREFIX + "_MODEL_DATA").columns)
    missing = set(["PATIENT_ID", "END_DT", "RESP"] + features).difference(columns)
    if missing:
        raise ValueError("MODEL_DATA is missing required columns: " + repr(sorted(missing)))
    return features, comparison

def fetch_source_features(features):
    fields = ["PATIENT_ID", "END_DT", "RESP"] + features
    selected = ", ".join("M." + quote_identifier(f) for f in fields)
    # Select only the frozen cohort. Duplicated source keys remain visible and fail validation.
    query = (f"SELECT {selected} FROM {DATABASE}.DS_ML.{SOURCE_PREFIX}_MODEL_DATA M "
             f"INNER JOIN (SELECT DISTINCT PATIENT_ID, END_DT FROM {DATABASE}.DS_ML.{PREFIX}_SNAPSHOTS) S "
             "ON M.PATIENT_ID = S.PATIENT_ID AND M.END_DT = S.END_DT")
    return read_query(query).toPandas()

def population_check(metadata):
    observed = (len(metadata), metadata.PATIENT_ID.nunique(), int(metadata.RESP.sum()))
    if observed != (23151, 12447, 1345):
        raise ValueError(f"Frozen V63 cohort changed: snapshots/patients/positives = {observed}")
import base64
def table_exists(table):
    if not re.fullmatch(r"[A-Z][A-Z0-9_]*", table):
        raise ValueError("Use uppercase letters, numbers and underscores in table names.")
    query = ("SELECT TABLE_NAME FROM DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.TABLES "
             f"WHERE TABLE_SCHEMA = 'DS_ML' AND TABLE_NAME = '{table}'")
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("query", query).load().limit(1).count() > 0)

def pack_artifacts(artifacts, chunk_size=50000):
    rows = []
    for name, blob in artifacts.items():
        encoded = base64.b64encode(blob).decode("ascii")
        pieces = [encoded[i:i + chunk_size] for i in range(0, len(encoded), chunk_size)] or [""]
        digest = hashlib.sha256(blob).hexdigest()
        rows.extend((name, i, len(pieces), len(blob), digest, piece)
                    for i, piece in enumerate(pieces))
    return rows

def unpack_artifacts(rows, expected_names):
    groups = {}
    for row in rows:
        name, i, count, size, digest, payload = tuple(row)
        if any(value != int(value) for value in (i, count, size)):
            raise ValueError("Nonintegral artifact chunk metadata.")
        groups.setdefault(name, []).append((int(i), int(count), int(size), digest, payload))
    if set(groups) != set(expected_names):
        raise ValueError("Missing or unexpected saved artifacts.")
    result = {}
    for name, pieces in groups.items():
        pieces.sort(key=lambda p: p[0])
        count, size, digest = pieces[0][1:4]
        if (count < 1 or size < 0 or len(pieces) != count
                or [p[0] for p in pieces] != list(range(count))
                or any(p[1:4] != (count, size, digest) for p in pieces)):
            raise ValueError("Missing, duplicate or inconsistent artifact chunks.")
        blob = base64.b64decode("".join(p[4] for p in pieces), validate=True)
        if len(blob) != size or hashlib.sha256(blob).hexdigest() != digest:
            raise ValueError("Artifact length/hash mismatch.")
        result[name] = blob
    return result

def read_artifacts(table, expected_names):
    rows = (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load().select(*ARTIFACT_COLUMNS).collect())
    return unpack_artifacts(rows, expected_names)

def save_artifacts(table, artifacts):
    # A matching existing result can be verified after an interrupted read-back.
    if table_exists(table):
        if read_artifacts(table, artifacts) != artifacts:
            raise FileExistsError("Destination contains different artifacts; choose a new RUN_ID.")
        print(f"Existing artifacts verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")
        return
    schema = ("ARTIFACT_NAME STRING, CHUNK_INDEX INT, CHUNK_COUNT INT, "
              "BYTE_LENGTH LONG, SHA256 STRING, PAYLOAD_BASE64 STRING")
    frame = spark.createDataFrame(pack_artifacts(artifacts), schema=schema)
    (frame.write.format("snowflake").options(**sf_options_dl_poc)
     .option("dbtable", table).option("truncate_columns", "off")
     .mode("errorifexists").save())
    if read_artifacts(table, artifacts) != artifacts:
        raise ValueError("Saved artifact read-back differs from the completed run.")
    print(f"Saved and verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")
ARTIFACT_COLUMNS = ["ARTIFACT_NAME", "CHUNK_INDEX", "CHUNK_COUNT", "BYTE_LENGTH", "SHA256", "PAYLOAD_BASE64"]

import io
import json
import hashlib
import numpy as np
import pandas as pd
"""Small sequence classifier; zero-activity months remain real timesteps."""

from dataclasses import dataclass

import torch
from torch import nn


@dataclass(frozen=True)
class ModelConfig:
    input_dim: int
    seq_len: int = 12
    d_model: int = 128
    n_heads: int = 4
    encoder_layers: int = 2
    feedforward_dim: int = 256
    dropout: float = 0.2

    def __post_init__(self):
        sizes = (self.input_dim, self.seq_len, self.d_model, self.n_heads,
                 self.encoder_layers, self.feedforward_dim)
        if any(not isinstance(n, int) or isinstance(n, bool) or n <= 0 for n in sizes):
            raise ValueError("All model dimensions must be positive integers.")
        if self.d_model % self.n_heads:
            raise ValueError("d_model must be divisible by n_heads.")
        if not 0 <= self.dropout < 1:
            raise ValueError("dropout must be in [0, 1).")


class ClaimsTransformer(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.projection = nn.Linear(config.input_dim, config.d_model)
        self.position = nn.Parameter(torch.empty(1, config.seq_len, config.d_model))
        nn.init.normal_(self.position, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=config.d_model, nhead=config.n_heads,
            dim_feedforward=config.feedforward_dim, dropout=config.dropout,
            activation="relu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            layer, num_layers=config.encoder_layers, enable_nested_tensor=False,
        )
        # TransformerEncoder clones the initial layer; initialize matrix weights
        # independently so the layers do not begin with identical weights.
        for encoder_layer in self.encoder.layers:
            for parameter in encoder_layer.parameters():
                if parameter.dim() > 1:
                    nn.init.xavier_uniform_(parameter)
        self.norm = nn.LayerNorm(config.d_model)
        self.head = nn.Sequential(
            nn.Linear(config.d_model, 64), nn.ReLU(),
            nn.Dropout(config.dropout), nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
        expected = (self.config.seq_len, self.config.input_dim)
        if x.ndim != 3 or tuple(x.shape[1:]) != expected or not x.is_floating_point():
            raise ValueError(f"Expected floating [batch, {expected[0]}, {expected[1]}] input.")
        if valid.dtype != torch.bool or tuple(valid.shape) != tuple(x.shape[:2]):
            raise ValueError("Validity must be a bool [batch, timestep] tensor; True means observed.")
        if valid.device != x.device or not torch.isfinite(x).all():
            raise ValueError("Inputs/masks must share a device and all inputs must be finite.")
        # PyTorch key-padding convention: True = IGNORE. Observed zeros remain valid.
        # https://docs.pytorch.org/docs/2.8/generated/torch.nn.Transformer.html
        # All-unavailable sequences retain their rows and use the existing head on
        # zero pooled history; no artificial clinical month is made observable.
        pooled = x.new_zeros((len(x), self.config.d_model))
        active = valid.any(dim=1)
        if active.any():
            row_valid = valid[active]
            clean = x[active].masked_fill(~row_valid.unsqueeze(-1), 0.)
            hidden = self.projection(clean) + self.position
            hidden = self.norm(self.encoder(hidden, src_key_padding_mask=~row_valid))
            # Key padding does not remove padded query outputs: exclude them here.
            hidden = hidden.masked_fill(~row_valid.unsqueeze(-1), 0.)
            pooled[active] = hidden.sum(dim=1) / row_valid.sum(dim=1, keepdim=True)
        return self.head(pooled).squeeze(-1)

"""Classification metrics and a threshold selected only on VALIDATION."""

import numpy as np
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)


def _validate(y, probabilities):
    labels = np.asarray(y)
    raw_scores = np.asarray(probabilities)
    if np.iscomplexobj(raw_scores) or (
        raw_scores.dtype == object
        and any(isinstance(value, (complex, np.complexfloating)) for value in raw_scores.flat)
    ):
        raise ValueError("Probabilities must be real values, not complex numbers.")
    scores = np.asarray(raw_scores, dtype=float)
    if labels.ndim != 1 or scores.ndim != 1 or len(labels) != len(scores) or not len(labels):
        raise ValueError("Labels and probabilities must be aligned, nonempty 1-D arrays.")
    if not np.isin(labels, [0, 1]).all():
        raise ValueError("Labels must be binary 0/1.")
    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError("Probabilities must be finite and in [0, 1].")
    return labels.astype(np.int64), scores


def select_validation_threshold(y, probabilities) -> float:
    """Maximize VALIDATION F1; an exact tie uses the highest threshold.

    The caller must provide VALIDATION labels and scores, never TEST. Predictions
    are positive when score >= threshold. Equal scores are never split, and
    integer cross-products identify exact F1 ties without rounding ambiguity.
    """
    labels, scores = _validate(y, probabilities)
    if len(np.unique(labels)) != 2:
        raise ValueError("Threshold selection requires both VALIDATION classes.")
    order = np.argsort(scores, kind="stable")[::-1]
    ranked_scores = scores[order]
    true_positives = np.cumsum(labels[order], dtype=np.int64)
    group_ends = np.r_[np.flatnonzero(ranked_scores[:-1] != ranked_scores[1:]), len(labels) - 1]
    total_positives = int(labels.sum())
    best_numerator, best_denominator = 0, 1
    best_threshold = float(ranked_scores[0])
    for end in group_ends:
        # F1 = 2 TP / (number selected + total positives). Python integers
        # keep cross-products exact and avoid fixed-width integer overflow.
        numerator = 2 * int(true_positives[end])
        denominator = int(end) + 1 + total_positives
        if numerator * best_denominator > best_numerator * denominator:
            best_numerator, best_denominator = numerator, denominator
            best_threshold = float(ranked_scores[end])
        # Descending thresholds retain the highest cutoff on an exact tie.
    return best_threshold


def classification_metrics(y, probabilities, threshold: float) -> dict:
    labels, scores = _validate(y, probabilities)
    if not np.isfinite(threshold) or not 0 <= threshold <= 1:
        raise ValueError("The fixed classification threshold must be in [0, 1].")
    predicted = (scores >= threshold).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
    return {
        "average_precision": float(average_precision_score(labels, scores)) if labels.sum() else None,
        "roc_auc": float(roc_auc_score(labels, scores)) if len(np.unique(labels)) == 2 else None,
        "precision": float(precision_score(labels, predicted, zero_division=0)),
        "recall": float(recall_score(labels, predicted, zero_division=0)),
        "f1": float(f1_score(labels, predicted, zero_division=0)),
        "threshold": float(threshold),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

"""Seed configuration and aggregate-only runtime provenance."""

import os
import platform
import random

import numpy as np
import sklearn
import torch


def seed_everything(seed: int) -> None:
    if not isinstance(seed, int) or isinstance(seed, bool) or not 0 <= seed < 2**32:
        raise ValueError("seed must be an integer in [0, 2**32).")
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def seed_worker(worker_id: int) -> None:
    """Use the DataLoader's seeded generator for each worker process."""
    del worker_id
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def resolve_device(device: str = "auto") -> torch.device:
    if device == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    resolved = torch.device(device)
    if resolved.type not in {"cpu", "cuda"}:
        raise ValueError("Supported devices are auto, cpu, or cuda[:index].")
    if resolved.type == "cuda" and not torch.cuda.is_available():
        raise ValueError("CUDA requested but unavailable; set device='cpu'.")
    return resolved


def runtime_metadata() -> dict:
    return {
        "python_version": platform.python_version(),
        "numpy_version": str(np.__version__),
        "sklearn_version": str(sklearn.__version__),
        "torch_version": str(torch.__version__),
        "cuda_version": str(torch.version.cuda),
        "deterministic_algorithms_requested": torch.are_deterministic_algorithms_enabled(),
        "determinism_scope": "Best effort within a fixed device and software environment; unsupported operations warn.",
    }


from dataclasses import asdict
from datetime import datetime, timezone
from torch.utils.data import Dataset, DataLoader

TIME_ORIENTATION = "0=most_recent; increasing_index=older"
MASK_CONVENTION = "valid=True means observed; src_key_padding_mask=~valid; masked mean pooling"


def _primitive_state(state):
    return json.loads(json.dumps(state, sort_keys=True, allow_nan=False))


def validate_model_inputs(data, require_splits=True):
    X, valid = data["X"], data["valid"]
    representation = data["representation"]
    steps = {"MONTHLY": 12, "QUARTERLY": 4}.get(representation)
    if steps is None or X.shape != (len(data["metadata"]), steps, 49):
        raise ValueError("Representation must be MONTHLY [N,12,49] or QUARTERLY [N,4,49].")
    if X.dtype != np.float32 or valid.dtype != np.bool_ or valid.shape != X.shape[:2]:
        raise ValueError("Expected float32 tensor and aligned boolean validity mask.")
    if not np.isfinite(X).all() or not np.all(X[~valid] == 0):
        raise ValueError("Inputs must be finite; padding must remain zero after preprocessing.")
    features = data["features"]
    if len(features) != 49 or len(set(features)) != 49 or "RESP" in [f.upper() for f in features]:
        raise ValueError("Exactly 49 unique predictors excluding RESP are required.")
    if not all(isinstance(f, str) and f for f in features):
        raise ValueError("Feature names must be nonempty strings.")
    meta = data["metadata"]
    if meta.duplicated(["PATIENT_ID", "END_DT"]).any():
        raise ValueError("Duplicate snapshot keys.")
    y = np.asarray(data["y"])
    if y.shape != (len(X),) or not np.isin(y, [0, 1]).all() or not np.array_equal(y, meta.RESP.to_numpy()):
        raise ValueError("Labels must match the original snapshot manifest exactly.")
    if not isinstance(data["hashes"], dict) or not data["hashes"]:
        raise ValueError("Validated source fingerprints are required.")
    _primitive_state(data["hashes"])
    _primitive_state(data["preprocessor"])
    if require_splits:
        indices = data["indices"]
        if set(indices) != {"train", "validation", "test"}:
            raise ValueError("All original patient split assignments are required.")
        seen = np.zeros(len(X), dtype=np.int8)
        sets = {}
        for split in ("train", "validation", "test"):
            rows = np.asarray(indices[split])
            if rows.ndim != 1 or not np.issubdtype(rows.dtype, np.integer) or not len(rows):
                raise ValueError("Each split requires nonempty integer snapshot indices.")
            if (rows < 0).any() or (rows >= len(X)).any() or len(np.unique(rows)) != len(rows):
                raise ValueError("Invalid or duplicate split indices.")
            seen[rows] += 1
            if not meta.iloc[rows].SPLIT.eq(split).all():
                raise ValueError("Loader assignments differ from the frozen split.")
            sets[split] = set(meta.iloc[rows].PATIENT_ID)
        if not np.all(seen == 1) or any(sets[a] & sets[b] for a, b in (("train", "validation"), ("train", "test"), ("validation", "test"))):
            raise ValueError("Population coverage or patient split separation failed.")
    return True


class SequenceDataset(Dataset):
    def __init__(self, data, split):
        self.data, self.indices = data, np.asarray(data["indices"][split])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        row = int(self.indices[index])
        x = torch.from_numpy(np.array(self.data["X"][row], dtype=np.float32, copy=True))
        valid = torch.from_numpy(np.array(self.data["valid"][row], dtype=bool, copy=True))
        y = torch.tensor(float(self.data["y"][row]), dtype=torch.float32)
        return x, valid, y


def make_loader(data, split, batch_size, seed, shuffle=False):
    return DataLoader(SequenceDataset(data, split), batch_size=batch_size,
                      shuffle=shuffle, num_workers=0, drop_last=False,
                      generator=torch.Generator().manual_seed(seed))


def predict_loader(model, loader, device, criterion=None):
    model.eval()
    labels, scores, total_loss, count = [], [], 0.0, 0
    with torch.inference_mode():
        for x, valid, y in loader:
            x, valid, y = x.to(device), valid.to(device), y.to(device)
            logits = model(x, valid)
            if not torch.isfinite(logits).all():
                raise ValueError("Nonfinite model predictions.")
            if criterion is not None:
                loss = criterion(logits, y)
                if not torch.isfinite(loss):
                    raise ValueError("Nonfinite evaluation loss.")
                total_loss += float(loss.item()) * len(y)
            count += len(y)
            labels.append(y.cpu().numpy())
            scores.append(torch.sigmoid(logits).cpu().numpy())
    if not count:
        raise ValueError("Cannot score an empty split.")
    return total_loss / count, np.concatenate(labels), np.concatenate(scores)


def rank_tables(metadata, scores):
    labels, scores = _validate(metadata.RESP.to_numpy(), scores)
    if metadata.duplicated(["PATIENT_ID", "END_DT"]).any() or metadata[["PATIENT_ID", "END_DT"]].isna().any().any():
        raise ValueError("Ranking requires unique nonnull snapshot keys.")
    ranked = metadata[["PATIENT_ID", "END_DT", "RESP"]].reset_index(drop=True).copy()
    ranked["SCORE"] = scores
    ranked = ranked.sort_values(["SCORE", "PATIENT_ID", "END_DT"], ascending=[False, True, True]).reset_index(drop=True)
    n, positives = len(ranked), int(labels.sum())
    base = positives / n
    ranked["DECILE"] = 10 - np.minimum(9, np.arange(n) * 10 // n)
    deciles = []
    cumulative_n = cumulative_positive = 0
    for decile in range(10, 0, -1):
        part = ranked[ranked.DECILE == decile]
        if part.empty:
            continue
        count, positive = len(part), int(part.RESP.sum())
        cumulative_n += count
        cumulative_positive += positive
        rate = positive / count
        deciles.append({"decile": decile, "snapshots": count, "positives": positive,
                        "score_min": float(part.SCORE.min()), "score_max": float(part.SCORE.max()),
                        "response_rate": rate, "lift": rate / base if base else None,
                        "cumulative_snapshots": cumulative_n, "cumulative_positives": cumulative_positive,
                        "cumulative_lift": (cumulative_positive / cumulative_n) / base if base else None,
                        "cumulative_recall": cumulative_positive / positives if positives else None})
    top = []
    for fraction in (.05, .10, .20, .30):
        k = max(1, int(np.ceil(n * fraction)))
        tp = int(ranked.RESP.iloc[:k].sum())
        top.append({"fraction": fraction, "observations_evaluated": n,
                    "baseline_positive_rate": base, "selected": k,
                    "positives": tp, "total_positives": positives,
                    "positive_rate_in_selected": tp / k, "resp1_captured": tp,
                    "precision": tp / k, "recall": tp / positives if positives else None,
                    "lift": (tp / k) / base if base else None})
    return pd.DataFrame(deciles), pd.DataFrame(top)


def top10_lift(y, scores, metadata):
    labels, scores = _validate(y, scores)
    if not np.array_equal(labels, metadata.RESP.to_numpy()) or not 0 < labels.sum() < len(labels):
        raise ValueError("Lift requires aligned metadata and both classes.")
    _, top = rank_tables(metadata, scores)
    return float(top.loc[top.fraction.eq(.10), "lift"].iloc[0])


def train_run(data, model_config, settings, run_id):
    validate_model_inputs(data)
    if tuple(data["X"].shape[1:]) != (model_config.seq_len, model_config.input_dim):
        raise ValueError("Architecture and tensor dimensions disagree.")
    for name in ("epochs", "patience", "batch_size"):
        if type(settings[name]) is not int or settings[name] <= 0:
            raise ValueError(f"{name} must be a positive integer.")
    for name in ("learning_rate", "grad_clip"):
        if not np.isfinite(settings[name]) or settings[name] <= 0:
            raise ValueError(f"{name} must be positive and finite.")
    for name in ("weight_decay", "min_delta"):
        if not np.isfinite(settings[name]) or settings[name] < 0:
            raise ValueError(f"{name} must be nonnegative and finite.")
    seed_everything(settings["seed"])
    device = resolve_device(settings["device"])
    class_counts = {}
    for name in ("train", "validation"):
        y = data["y"][data["indices"][name]]
        positives = int(y.sum())
        negatives = len(y) - positives
        if not positives or not negatives:
            raise ValueError(f"{name} requires both classes.")
        class_counts[name] = {"snapshots": len(y), "positives": positives, "negatives": negatives}
    pos_weight = class_counts["train"]["negatives"] / class_counts["train"]["positives"]
    model = ClaimsTransformer(model_config).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=settings["learning_rate"], weight_decay=settings["weight_decay"])
    train_loader = make_loader(data, "train", settings["batch_size"], settings["seed"], True)
    validation_loader = make_loader(data, "validation", settings["batch_size"], settings["seed"])
    diagnostic_loader = make_loader(data, "train", settings["batch_size"], settings["seed"])
    best_ap, patience_reference = -np.inf, -np.inf
    without_progress, best_epoch, best_state = 0, None, None
    history = []
    stop_reason, early_stopping_epoch = "maximum_epochs", None
    print(f'{data["representation"]}: training on {device}; TRAIN positive weight={pos_weight:.4f}.', flush=True)
    for epoch in range(1, settings["epochs"] + 1):
        model.train()
        loss_sum, count = 0.0, 0
        for x, valid, y in train_loader:
            x, valid, y = x.to(device), valid.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(x, valid), y)
            if not torch.isfinite(loss):
                raise ValueError("Nonfinite training loss.")
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), settings["grad_clip"], error_if_nonfinite=True)
            optimizer.step()
            loss_sum += float(loss.item()) * len(y)
            count += len(y)
        val_loss, val_y, val_scores = predict_loader(model, validation_loader, device, criterion)
        val_ap = float(average_precision_score(val_y, val_scores))
        train_loss, train_y, train_scores = predict_loader(model, diagnostic_loader, device, criterion)
        train_ap = float(average_precision_score(train_y, train_scores))
        train_lift = top10_lift(train_y, train_scores, data["metadata"].iloc[data["indices"]["train"]])
        val_lift = top10_lift(val_y, val_scores, data["metadata"].iloc[data["indices"]["validation"]])
        history.append({"epoch": epoch, "optimization_loss": loss_sum / count,
                        "training_loss": train_loss, "validation_loss": val_loss,
                        "training_top10_lift": train_lift, "validation_top10_lift": val_lift,
                        "training_average_precision": train_ap, "validation_average_precision": val_ap,
                        "validation_roc_auc": float(roc_auc_score(val_y, val_scores))})
        if val_ap > best_ap:
            best_ap, best_epoch = val_ap, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if val_ap > patience_reference + settings["min_delta"]:
            patience_reference, without_progress = val_ap, 0
        else:
            without_progress += 1
        print(f'Epoch {epoch:02d}: train loss={train_loss:.5f}; validation loss={val_loss:.5f}; '
              f'TRAIN AP={train_ap:.5f}; VALIDATION AP={val_ap:.5f}; '
              f'TRAIN lift={train_lift:.3f}; VALIDATION lift={val_lift:.3f}', flush=True)
        if without_progress >= settings["patience"]:
            stop_reason, early_stopping_epoch = "validation_AP_patience", epoch
            print(f"Early stopping; restoring epoch {best_epoch}.", flush=True)
            break
    if best_state is None:
        raise ValueError("Training did not produce a valid checkpoint.")
    model.load_state_dict(best_state, strict=True)
    _, val_y, val_scores = predict_loader(model, validation_loader, device, criterion)
    if not np.isclose(average_precision_score(val_y, val_scores), best_ap, rtol=0, atol=1e-7):
        raise ValueError("Restored checkpoint failed validation AP reproduction.")
    threshold = select_validation_threshold(val_y, val_scores)
    state = _primitive_state(data["preprocessor"])
    summary = {"run_id": run_id, "representation": data["representation"], "training_complete": True,
               "completed_at_utc": datetime.now(timezone.utc).isoformat(),
               "best_epoch": int(best_epoch), "epochs_completed": len(history),
               "early_stopping_epoch": early_stopping_epoch, "stop_reason": stop_reason,
               "best_validation_average_precision": best_ap, "validation_threshold": threshold,
               "validation_metrics": classification_metrics(val_y, val_scores, threshold),
               "train_pos_weight": float(pos_weight), "class_counts": class_counts,
               "model_parameter_count": sum(p.numel() for p in model.parameters()),
               "model_config": asdict(model_config), "training_settings": dict(settings),
               "input_hashes": _primitive_state(data["hashes"]), "resolved_device": str(device),
               "preprocessor": state, "time_orientation": TIME_ORIENTATION, "mask_convention": MASK_CONVENTION,
               "all_padded_snapshots": int((~data["valid"].any(axis=1)).sum()),
               "preprocessing": "Already transformed using frozen TRAIN-fitted preprocessing; no additional log1p",
               "test_inference_performed": False, "runtime": runtime_metadata()}
    payload = {"format_version": 2, "training_complete": True, "run_id": run_id,
               "representation": data["representation"], "model_state_dict": best_state,
               "model_config": asdict(model_config), "training_settings": _primitive_state(settings),
               "input_hashes": _primitive_state(data["hashes"]), "feature_names": list(data["features"]),
               "time_steps": list(range(model_config.seq_len)), "time_orientation": TIME_ORIENTATION,
               "mask_convention": MASK_CONVENTION, "preprocessor": state,
               "tensor_shape": [int(v) for v in data["X"].shape],
               "transform": "frozen_train_preprocessor", "validation_threshold": float(threshold),
               "selection_metric": "validation_average_precision", "best_epoch": int(best_epoch),
               "best_validation_average_precision": best_ap}
    buffer = io.BytesIO()
    torch.save(payload, buffer)
    return buffer.getvalue(), summary, pd.DataFrame(history)


def load_verified_model(blob, data, run_id, device="auto", expected_settings=None, expected_config=None):
    validate_model_inputs(data)
    payload = torch.load(io.BytesIO(blob), map_location="cpu", weights_only=True)
    if (not isinstance(payload, dict) or payload.get("format_version") != 2
            or payload.get("training_complete") is not True or payload.get("run_id") != run_id):
        raise ValueError("Checkpoint is incomplete, unsupported, or belongs to another run.")
    expected = {"input_hashes": _primitive_state(data["hashes"]),
                "feature_names": list(data["features"]), "representation": data["representation"],
                "preprocessor": _primitive_state(data["preprocessor"]),
                "time_steps": list(range(data["X"].shape[1])), "tensor_shape": list(data["X"].shape),
                "time_orientation": TIME_ORIENTATION, "mask_convention": MASK_CONVENTION,
                "transform": "frozen_train_preprocessor", "selection_metric": "validation_average_precision"}
    for key, value in expected.items():
        if payload.get(key) != value:
            raise ValueError(f"Checkpoint differs from verified inputs on {key}.")
    if expected_settings is not None and payload.get("training_settings") != _primitive_state(expected_settings):
        raise ValueError("Checkpoint training settings differ from declared settings.")
    if expected_config is not None:
        expected_config = asdict(expected_config) if isinstance(expected_config, ModelConfig) else dict(expected_config)
        if payload.get("model_config") != expected_config:
            raise ValueError("Checkpoint architecture differs from declared settings.")
    threshold = payload.get("validation_threshold")
    if threshold is None or not np.isfinite(threshold) or not 0 <= threshold <= 1:
        raise ValueError("Missing valid frozen validation threshold.")
    model = ClaimsTransformer(ModelConfig(**payload["model_config"]))
    if (model.config.seq_len, model.config.input_dim) != tuple(data["X"].shape[1:]):
        raise ValueError("Checkpoint architecture is inconsistent with tensor layout.")
    model.load_state_dict(payload["model_state_dict"], strict=True)
    resolved = resolve_device(device)
    model.to(resolved).eval()
    return model, payload, resolved


def plot_history(history, title):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    history.plot(x="epoch", y=["training_loss", "validation_loss"], ax=axes[0], title=title + " loss (eval mode)")
    history.plot(x="epoch", y=["training_average_precision", "validation_average_precision"], ax=axes[1], title=title + " AP")
    history.plot(x="epoch", y=["training_top10_lift", "validation_top10_lift"], ax=axes[2], title=title + " top-10% lift")
    fig.tight_layout()
    return fig


def model_contract(data, config):
    return {'implementation_sha256': IMPLEMENTATION_SHA256, 'input_hashes': data['hashes'],
            'representation': data['representation'], 'config': asdict(config), 'training': TRAINING_SETTINGS}

def restored_representation(name, data):
    config = ModelConfig(input_dim=49, seq_len=data['X'].shape[1], **MODEL_SETTINGS)
    blobs = read_artifacts(RUN_PREFIX + '_' + name + '_MODEL', MODEL_NAMES)
    summary = json.loads(blobs['summary.json'])
    require(summary['contract'] == model_contract(data, config), 'Saved checkpoint contract differs.')
    model, payload, device = load_verified_model(blobs['checkpoint.pt'], data, RUN_ID + '_' + name,
        expected_settings=TRAINING_SETTINGS, expected_config=config)
    require(summary['best_epoch'] == payload['best_epoch'] and summary['best_validation_average_precision'] == payload['best_validation_average_precision'], 'Summary/checkpoint selection differs.')
    return model, payload, device, summary, pd.DataFrame(json.loads(blobs['history.json']))

def evaluate_partition(model, payload, data, split, device):
    rows = data['indices'][split]
    _, y, scores = predict_loader(model, make_loader(data, split, TRAINING_SETTINGS['batch_size'], 42), device)
    meta = data['metadata'].iloc[rows].reset_index(drop=True)
    require(np.array_equal(y, meta.RESP.to_numpy()), 'Prediction labels do not align with snapshot keys.')
    deciles, top = rank_tables(meta, scores)
    metrics = classification_metrics(y, scores, payload['validation_threshold'])
    top10 = top.loc[top.fraction.eq(.1)].iloc[0]
    metrics.update(model=data['representation'], split=split, snapshots=len(y), positives=int(y.sum()),
        top10_lift=float(top10['lift']), top10_precision=float(top10['precision']), top10_recall=float(top10['recall']))
    if split == 'validation':
        require(np.isclose(metrics['average_precision'], payload['best_validation_average_precision'], rtol=0, atol=1e-7), 'Saved validation AP failed reproduction.')
    return {'metrics': metrics, 'deciles': deciles, 'top': top}


In [ ]:
# Load verified inputs, masks, preprocessing and frozen assignments
experiments, manifest, split_audit = load_experiment()
display(pd.DataFrame([manifest['representations'][name]['sparsity'] for name in ('MONTHLY', 'QUARTERLY')]))
display(pd.DataFrame(manifest['audit']))
print('TEST was previously inspected; these are retrospective results, not untouched confirmation.')


In [ ]:
# Evaluate MONTHLY and QUARTERLY independently with frozen checkpoints
reports, summaries, histories = {}, {}, {}
for name in ('MONTHLY', 'QUARTERLY'):
    data = experiments[name]
    model, payload, device, summary, history = restored_representation(name, data)
    reports[name], summaries[name], histories[name] = {}, summary, history
    for split in ('train', 'validation', 'test'):
        report = evaluate_partition(model, payload, data, split, device)
        reports[name][split] = report
        print(name, split.upper(), '| top 10% of snapshots; ties ordered by PATIENT_ID then END_DT')
        display(pd.DataFrame([report['metrics']]))
        display(report['top'].loc[report['top'].fraction.eq(.1)])
        display(report['deciles'])
        display(report['top'])
    del model


In [ ]:
# Display the requested final comparison table and train-validation gaps
comparison_rows, gap_rows = [], []
for name in ('MONTHLY', 'QUARTERLY'):
    tr, va, te = [reports[name][s]['metrics'] for s in ('train', 'validation', 'test')]
    comparison_rows.append({'MODEL': name + ' TRANSFORMER', 'TIMESTEPS': experiments[name]['X'].shape[1],
        'FEATURE_COUNT': 49, 'PADDING/MASKING': 'Zero padding; key-padding mask and masked mean',
        'TRAIN_AP': tr['average_precision'], 'VALIDATION_AP': va['average_precision'],
        'TEST_AP': te['average_precision'], 'TEST_AUC': te['roc_auc'], 'TEST_TOP10_LIFT': te['top10_lift'],
        'TEST_TOP10_PRECISION': te['top10_precision'], 'TEST_TOP10_RECALL': te['top10_recall']})
    gap_rows.append({'MODEL': name, 'TRAIN_MINUS_VALIDATION_AP': tr['average_precision'] - va['average_precision'],
                     'TRAIN_MINUS_VALIDATION_LIFT': tr['top10_lift'] - va['top10_lift'],
                     'BEST_EPOCH': summaries[name]['best_epoch'], 'STOPPED_AT_EPOCH': summaries[name]['epochs_completed'],
                     'EARLY_STOPPING_EPOCH': summaries[name]['early_stopping_epoch']})
comparison = pd.DataFrame(comparison_rows)
display(comparison)
display(pd.DataFrame(gap_rows))
print('LightGBM omitted: no verified same-population benchmark artifact is configured in this repository.')
print('AP here is sklearn average_precision_score, not trapezoidal PR area.')


In [ ]:
# Display loss, AP and lift histories and retrospective lift curves
import matplotlib.pyplot as plt
for name in ('MONTHLY', 'QUARTERLY'):
    plot_history(histories[name], name)
    plt.show()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for split, report in reports[name].items():
        axes[0].plot(report['deciles'].decile, report['deciles'].lift, marker='o', label=split)
        axes[1].plot(report['top'].fraction * 100, report['top'].lift, marker='o', label=split)
    axes[0].invert_xaxis()
    axes[0].set(xlabel='Decile (10 = highest)', ylabel='Lift', title=name)
    axes[1].set(xlabel='Top % of snapshots', ylabel='Lift', title=name)
    for ax in axes:
        ax.legend()
        ax.grid(alpha=.3)
    plt.show()


In [ ]:
# Save aggregate results privately and display factual execution summary
result = {'run_id': RUN_ID, 'manifest_sha256': digest_json(manifest),
    'comparison': comparison.to_dict('records'), 'gaps': gap_rows,
    'metrics': {name: {split: item['metrics'] for split, item in group.items()} for name, group in reports.items()},
    'test_retrospective': True, 'causal_masking_effect_established': False}
save_artifacts(RUN_PREFIX + '_EVALUATION', {'evaluation.json': canonical_json(result).encode()})
audit = pd.DataFrame(manifest['audit'])
counts = audit.HISTORICAL_RECONSTRUCTION_STATUS.value_counts().reindex(['EXACT', 'APPROXIMATED', 'SNAPSHOT_ONLY', 'UNRESOLVED'], fill_value=0)
print('Confirmed features:', len(manifest['features']))
print('Reconstruction counts:', counts.to_dict())
for name in ('MONTHLY', 'QUARTERLY'):
    print(name, '| actual shape:', experiments[name]['X'].shape)
    display(pd.DataFrame([manifest['representations'][name]['sparsity']]))
display(comparison)
display(audit.loc[audit.HISTORICAL_RECONSTRUCTION_STATUS.isin(['SNAPSHOT_ONLY', 'UNRESOLVED', 'APPROXIMATED'])])
print('Overfitting change versus the previous run: inconclusive without a verified comparable previous-run diagnostic report.')
print('Use displayed held-out metrics and histories to assess these hypotheses; no superiority or padding benefit is assumed.')
